# Capacity Accreditation

How much capacity should a resource that is already in an energy system be credited with? `assetra` has two marginal accreditation metrics for this, `MarginalReliabilityImpact` (MRI) and `MarginalEffectiveLoadCarryingCapability` (marginal ELCC). Both enlarge a resource by a small step, measure how resource adequacy responds, and return a dimensionless *accreditation factor*. They use the resource's own sampled outages, unlike `EffectiveLoadCarryingCapability`, which quantifies the contribution of an addition to a system and returns power.

Both metrics follow the gradient-based framework of Q. Zhang, F. Zhao, G. Stephen, C. Singh and L. Xie, "A Gradient-Based Capacity Accreditation Framework in Resource Adequacy: Formulation, Computation, and Practical Implications," [arXiv:2601.22087](https://arxiv.org/abs/2601.22087), 2026, referred to below as the paper. This notebook uses a small synthetic system, so its factors illustrate the methods and say nothing about any real system.

This notebook covers:

1. Building the system: one winter week with thermal, wind, solar, storage and hydro resources.
2. The marginal reliability impact of one thermal unit, by forward and by central difference.
3. The marginal ELCC of the same unit, by bisection and by secant, and what `perturbation_count` reports.
4. All four methods side by side for several resources, in the spirit of Table I of the paper.
5. The default step size versus an absolute step of 10 MW.
6. Expected unserved energy versus loss of load hours, and why counting metrics need a larger step.

## 1. Build the system

The system covers one winter week: 168 hours from 1 January, simulated with 200 Monte Carlo trials. Powers are in MW and energies in MWh. It contains

- six thermal units, each a `StochasticUnit`, with different nameplates and forced outage rates,
- a wind farm and a solar farm, each a `StaticUnit`, whose output is the same in every trial,
- one 2-hour battery, a `StorageUnit`, lossless (`roundtrip_efficiency=1.0`) for simplicity,
- one `HydroUnit`, and
- the demand, a `DemandUnit`.

Demand follows a daily shape with a morning and an evening peak. A cold snap on days 4 and 5 raises demand and calms the wind. The system is deliberately tight, so that shortfalls are frequent enough to measure with only 200 trials; a real study would use many more.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from assetra.contribution import (
    MarginalEffectiveLoadCarryingCapability,
    MarginalReliabilityImpact,
)
from assetra.metrics import ExpectedUnservedEnergy, LossOfLoadHours
from assetra.simulation import ProbabilisticSimulation
from assetra.system import EnergySystemBuilder
from assetra.units import (
    DemandUnit,
    HydroUnit,
    StaticUnit,
    StochasticUnit,
    StorageUnit,
)
from assetra.utils import get_hourly_time_series_xr

SEED = 2019
np.random.seed(SEED)

HOURS = 168  # h, one week
TRIAL_SIZE = 200
SIMULATION_START_HOUR = "2019-01-01 00:00:00"
SIMULATION_END_HOUR = "2019-01-07 23:00:00"

The profiles are synthetic. Energy units take hourly profiles as time-indexed `xarray.DataArray` objects, and `get_hourly_time_series_xr` builds one from a plain list of hourly values.

In [ ]:
def hourly(values):
    """Return hourly values as a time-indexed DataArray for the simulated week."""
    return get_hourly_time_series_xr(values, start_hour=SIMULATION_START_HOUR)


hour_of_day = np.arange(HOURS) % 24
day = np.arange(HOURS) // 24
cold_snap = (day == 3) | (day == 4)  # days 4 and 5

# demand: a daily shape (fraction of the peak), higher during the cold snap
PEAK_DEMAND = 600  # MW
DAILY_DEMAND_SHAPE = np.array(
    [0.66, 0.63, 0.62, 0.62, 0.64, 0.70, 0.80, 0.90, 0.92, 0.88, 0.85, 0.83,
     0.82, 0.81, 0.80, 0.82, 0.88, 0.97, 1.00, 0.98, 0.92, 0.84, 0.76, 0.69]
)
demand = (
    PEAK_DEMAND * DAILY_DEMAND_SHAPE[hour_of_day] * np.where(cold_snap, 1.06, 0.95)
)

# wind capacity factor: a slow swing over several days, calm during the cold snap
wind_capacity_factor = (
    0.45
    + 0.20 * np.sin(2 * np.pi * np.arange(HOURS) / 61 + 0.8)
    + 0.08 * np.sin(2 * np.pi * hour_of_day / 24 + 2.0)
)
wind_capacity_factor = np.where(
    cold_snap, 0.35 * wind_capacity_factor, wind_capacity_factor
).clip(0.02, 0.95)

# solar capacity factor: short winter days, from 8:00 to 16:00
solar_capacity_factor = 0.55 * np.clip(
    np.sin(np.pi * (hour_of_day - 7.5) / 9), 0, None
)

Every unit needs a unique `id`, which the accreditation classes use to find a resource in the system. A `HydroUnit` takes a nameplate and the expected generation of each month: assetra spreads the monthly energy over the simulated hours of that month, in proportion to the net demand left after the demand and static units, so here all 4000 MWh of January are dispatched within the simulated week, 24 MW on average.

In [ ]:
# thermal units: (id, nameplate in MW, forced outage rate)
THERMAL_SPECS = [
    (1, 150, 0.06),
    (2, 130, 0.05),
    (3, 110, 0.04),
    (4, 90, 0.08),
    (5, 70, 0.10),
    (6, 50, 0.12),
]

thermal_units = [
    StochasticUnit(
        id=unit_id,
        nameplate_capacity=nameplate,
        hourly_capacity=hourly([nameplate] * HOURS),
        hourly_forced_outage_rate=hourly([forced_outage_rate] * HOURS),
    )
    for unit_id, nameplate, forced_outage_rate in THERMAL_SPECS
]
wind = StaticUnit(
    id=7, nameplate_capacity=150, hourly_capacity=hourly(150 * wind_capacity_factor)
)
solar = StaticUnit(
    id=8, nameplate_capacity=100, hourly_capacity=hourly(100 * solar_capacity_factor)
)
battery = StorageUnit(
    id=9,
    nameplate_capacity=50,
    charge_rate=50,
    discharge_rate=50,
    charge_capacity=100,  # MWh
    roundtrip_efficiency=1.0,
)
hydro = HydroUnit(
    id=10,
    nameplate_capacity=60,
    monthly_expected_generation=xr.DataArray(
        [4000.0] * 12, coords=dict(month=list(range(1, 13)))  # MWh
    ),
)

builder = EnergySystemBuilder()
builder.add_unit(DemandUnit(id=0, hourly_demand=hourly(demand)))
for unit in [*thermal_units, wind, solar, battery, hydro]:
    builder.add_unit(unit)

energy_system = builder.build()
energy_system

A baseline simulation shows the risk of this system. Both expected unserved energy (EUE) and loss of load hours (LOLH) are above zero, which the accreditation classes need (for a system with no risk they raise a `RuntimeWarning`), and the shortfalls cluster on the cold-snap days.

In [ ]:
simulation = ProbabilisticSimulation(
    SIMULATION_START_HOUR, SIMULATION_END_HOUR, TRIAL_SIZE
)
simulation.assign_energy_system(energy_system)
simulation.run()

print(f"EUE:  {ExpectedUnservedEnergy(simulation).evaluate():.1f} MWh per week")
print(f"LOLH: {LossOfLoadHours(simulation).evaluate():.2f} h per week")

# share of the hours of each day that are in shortfall
shortfall = simulation.net_hourly_capacity_matrix < 0
shortfall_by_day = shortfall.groupby("time.day").mean(dim=["trial", "time"])
print("share of hours in shortfall, by day (%):", np.round(100 * shortfall_by_day.values, 1))

## 2. Marginal reliability impact of one unit

The accreditation classes take the base `EnergySystem`, a `ProbabilisticSimulation` that sets the study period and the trial size, and a resource adequacy metric class. The simulation is only used for its settings: the classes copy it and never modify it. `evaluate()` then takes the `resources` to accredit as an `EnergySystem` of units that are already in the base system. A convenient way to get one is to build it with an `EnergySystemBuilder` from the same unit objects: units are matched to the base system by type and id, and the base system's data is used. Several units are accredited together as one resource.

`MarginalReliabilityImpact` enlarges the resource by a step Δ, 1% of its nameplate by default, and compares the resulting change in EUE with the change that a perfect resource of the same size would bring, which adds Δ in every trial and hour. Let M(Δ) be the EUE with the resource enlarged by Δ (reduced, for a negative Δ), P(Δ) the EUE with a perfect resource of size Δ added instead, and M0 the baseline EUE. The `method` keyword selects

| `method` | Accreditation factor | N_p |
| --- | --- | --- |
| `"forward"` | (M(+Δ) - M0) / (P(+Δ) - M0) | 1 |
| `"central"` | (M(+Δ) - M(-Δ)) / (P(+Δ) - P(-Δ)) | 2 |

N_p is the number of simulations with the resource perturbed, the paper's measure of computational cost. The baseline and perfect-resource simulations are not counted. After each call, `perturbation_count` reports N_p.

Each `evaluate()` call samples the outages of the system afresh, once, and every simulation of the call reuses those samples, including the resource's own outages. We seed NumPy's random number generator right before each call. That makes the notebook reproducible, and lets the methods that we compare see the same outages.

In [ ]:
def system_of(*units):
    """Return an EnergySystem of units that are already in the base system."""
    builder = EnergySystemBuilder()
    for unit in units:
        builder.add_unit(unit)
    return builder.build()


# the 110 MW thermal unit, with a forced outage rate of 4%
thermal_unit = system_of(thermal_units[2])

mri_forward = MarginalReliabilityImpact(
    energy_system, simulation, ExpectedUnservedEnergy, method="forward"
)
mri_central = MarginalReliabilityImpact(
    energy_system, simulation, ExpectedUnservedEnergy, method="central"
)

for name, mri in [("forward", mri_forward), ("central", mri_central)]:
    np.random.seed(SEED)
    factor = mri.evaluate(thermal_unit)
    accredited = factor * thermal_unit.system_capacity  # MW
    print(
        f"MRI {name}: factor {factor:.3f}, N_p {mri.perturbation_count}, "
        f"accredited {accredited:.1f} of {thermal_unit.system_capacity:.0f} MW"
    )

`evaluate()` returns the factor, which is dimensionless. Multiplied by the nameplate capacity of the resources, `resources.system_capacity`, it gives their accredited capacity in MW, as printed above. (`EffectiveLoadCarryingCapability`, in contrast, returns power.)

The unit's forced outage rate is 4%, so it is available 96% of the time, and its factor is lower than that. The unit's own outages are one of the reasons the system falls short, so in the hours that matter it is out more often than on average. This is why the accreditation uses the resource's own sampled outages and not an independent copy of them.

## 3. Marginal ELCC of the same unit

`MarginalEffectiveLoadCarryingCapability` asks a different question: how much constant load can the system carry, with the resource enlarged by the step, while its EUE stays no worse than the baseline? That load is found by a root search, and the factor is the load divided by the step. The `method` keyword selects the search:

- `"bisection"` halves a bracket of loads until it is no wider than the resolution, which is `additional_demand_resolution_pct` times the step. At its default of 0.01 the factor is accurate to about 0.005. When the bracket from 0 to the step is known in advance to hold the load, which here is the case for EUE and resources without storage, N_p is 7.
- `"secant"` is the secant method of the paper, safeguarded by bisection steps. Its N_p depends on the data.

N_p again counts the simulations with the resource perturbed; the baseline simulation is not counted. The search needs a metric whose risk never decreases as load is added, so the marginal ELCC accepts `ExpectedUnservedEnergy`, `LossOfLoadHours` and `LossOfLoadDays`, while `MarginalReliabilityImpact` accepts any metric.

In [ ]:
elcc_bisection = MarginalEffectiveLoadCarryingCapability(
    energy_system, simulation, ExpectedUnservedEnergy, method="bisection"
)
elcc_secant = MarginalEffectiveLoadCarryingCapability(
    energy_system, simulation, ExpectedUnservedEnergy, method="secant"
)

for name, elcc in [("bisection", elcc_bisection), ("secant", elcc_secant)]:
    np.random.seed(SEED)
    factor = elcc.evaluate(thermal_unit)
    accredited = factor * thermal_unit.system_capacity  # MW
    print(
        f"ELCC {name}: factor {factor:.3f}, N_p {elcc.perturbation_count}, "
        f"accredited {accredited:.1f} of {thermal_unit.system_capacity:.0f} MW"
    )

Under EUE the two metrics estimate the same quantity, which the paper shows to coincide in the limit of a small step, and with the same outages the four factors above agree closely. They differ in cost: MRI needs one or two simulations with the resource perturbed, while the marginal ELCC needs a root search.

## 4. Comparing methods and resources

The resources can be a single unit, as above, or a group of units that is accredited as one resource, such as a whole fleet. `energy_system.get_system_by_type(StochasticUnit)` returns the thermal fleet as one `EnergySystem`, whose six units are enlarged together. Constructing a method runs no simulation, and each method can be evaluated on any number of resources. Two helpers follow: one creates the four methods for a metric class and a step, the other evaluates a method with the seed, as before.

The table has one row per resource and, for each method, its factor and N_p. It is in the spirit of Table I of the paper, which compares the accreditation factors of generators of several types under ELCC and MRI methods; here N_p is shown next to each factor.

In [ ]:
def make_methods(resource_adequacy_metric, perturbation_size=None):
    """Return the four accreditation methods for a metric class and a step size."""
    return {
        "MRI forward": MarginalReliabilityImpact(
            energy_system,
            simulation,
            resource_adequacy_metric,
            perturbation_size=perturbation_size,
            method="forward",
        ),
        "MRI central": MarginalReliabilityImpact(
            energy_system,
            simulation,
            resource_adequacy_metric,
            perturbation_size=perturbation_size,
            method="central",
        ),
        "ELCC bisection": MarginalEffectiveLoadCarryingCapability(
            energy_system,
            simulation,
            resource_adequacy_metric,
            perturbation_size=perturbation_size,
            method="bisection",
        ),
        "ELCC secant": MarginalEffectiveLoadCarryingCapability(
            energy_system,
            simulation,
            resource_adequacy_metric,
            perturbation_size=perturbation_size,
            method="secant",
        ),
    }


def accredit(method, resources):
    """Return the factor and N_p of one evaluate() call, seeded as above."""
    np.random.seed(SEED)
    factor = method.evaluate(resources)
    return factor, method.perturbation_count

In [ ]:
resources = {
    "Thermal unit 3 (110 MW)": thermal_unit,
    "Thermal fleet (6 units)": energy_system.get_system_by_type(StochasticUnit),
    "Wind farm": system_of(wind),
    "Solar farm": system_of(solar),
    "Battery": system_of(battery),
    "Hydro unit": system_of(hydro),
}

methods = make_methods(ExpectedUnservedEnergy)

columns = pd.MultiIndex.from_tuples(
    [("", "Nameplate (MW)")]
    + [(name, column) for name in methods for column in ("factor", "N_p")]
)
rows = []
for resource in resources.values():
    row = [resource.system_capacity]
    for method in methods.values():
        row.extend(accredit(method, resource))
    rows.append(row)

summary = pd.DataFrame(rows, index=list(resources), columns=columns)
summary.round(3)

All four methods give nearly the same factor for every resource, within about 0.01 here, while their costs differ. N_p is 1 for the forward and 2 for the central MRI. Bisection takes 7 simulations at the default resolution, and more for the battery, whose bracket is not known in advance. The secant's N_p depends on the data, and is not always below the bisection's.

The factors differ much more between resources than between methods, because a resource earns credit for what it produces when the system is short. For the wind farm, the solar farm and the hydro unit, whose output is the same in every trial, this can be checked directly: weight each hour's capacity factor by the share of trials in which that hour is in shortfall.

In [ ]:
shortfall_share = shortfall.mean("trial").values  # by hour
hydro_output = simulation.get_hourly_capacity_matrix_by_type(HydroUnit).isel(trial=0)

capacity_factors = {
    "Wind farm": wind_capacity_factor,
    "Solar farm": solar_capacity_factor,
    "Hydro unit": hydro_output.values / hydro.nameplate_capacity,
}
for name, capacity_factor in capacity_factors.items():
    weighted = np.average(capacity_factor, weights=shortfall_share)
    print(
        f"{name}: capacity factor {capacity_factor.mean():.3f} on average, "
        f"{weighted:.3f} in shortfall hours"
    )

The weighted capacity factors are close to the factors in the table. The wind farm produces about half of its average output in shortfall hours, because the calm cold snap is when the system is short. The solar farm produces far less than its average, because demand peaks in the morning and evening, when the sun is low or down. The hydro unit dispatches in proportion to net demand, so it produces more than its average when the system is short.

The thermal factors are below the availability of the units, 96% for thermal unit 3 and 93%, weighted by nameplate, for the fleet, because their own outages are among the causes of the shortfall. The battery is credited for what it discharges when the system is short. Its factor is high but below 1, as a battery can only discharge while it has energy left, and this one holds two hours' worth.

## 5. The default step versus a 10 MW step

By default the step is 1% of the nameplate of the resources passed to `evaluate()`: 1.1 MW for the thermal unit, 6 MW for the thermal fleet, 1.5 MW for the wind farm. `perturbation_size` sets an absolute step instead, in the power units of the system, so `perturbation_size=10` is the 10 MW step used in the paper's experiments. An absolute step perturbs every resource by the same amount, while the default step scales with each resource. With `method="central"` the step cannot exceed the nameplate of the resources.

The table compares the two steps for the MRI by forward difference and the ELCC by bisection.

In [ ]:
methods_10mw = make_methods(ExpectedUnservedEnergy, perturbation_size=10)
chosen = ["MRI forward", "ELCC bisection"]

columns = pd.MultiIndex.from_tuples(
    [("Default step", "MW")]
    + [(name, step) for name in chosen for step in ("default", "10 MW")]
)
rows = []
for resource in resources.values():
    row = [0.01 * resource.system_capacity]
    for name in chosen:
        for step_methods in (methods, methods_10mw):
            row.append(accredit(step_methods[name], resource)[0])
    rows.append(row)

pd.DataFrame(rows, index=list(resources), columns=columns).round(3)

For EUE the step changes the factors by at most a few hundredths here, much less than the differences between resources, so the default is a reasonable choice. MRI and the marginal ELCC agree in the limit of a small step, so the step should stay small relative to the resource. For counting metrics it also has to be large enough, as the next section shows.

## 6. Counting metrics: EUE versus loss of load hours

`MarginalReliabilityImpact` accepts any resource adequacy metric, and the marginal ELCC accepts `ExpectedUnservedEnergy`, `LossOfLoadHours` and `LossOfLoadDays`. These measure different things: EUE counts the energy that is not served, while LOLH counts the hours with a shortfall, so the same resource can earn different factors from them.

Counting metrics also need a larger step than EUE. LOLH changes only when an hour moves out of shortfall, or into it, and a small step moves few hours. The factor then rests on a handful of hours and is noisy, and if the step moves no hour at all, the metric does not respond and the classes raise a `RuntimeWarning` instead of returning a number. Here a step of 10 MW moves enough hours. The first table shows the wind farm under both metrics with that step.

In [ ]:
wind_farm = resources["Wind farm"]
eue_methods = make_methods(ExpectedUnservedEnergy, perturbation_size=10)
lolh_methods = make_methods(LossOfLoadHours, perturbation_size=10)

rows = []
for name in eue_methods:
    rows.append(
        [*accredit(eue_methods[name], wind_farm), *accredit(lolh_methods[name], wind_farm)]
    )

pd.DataFrame(
    rows,
    index=list(eue_methods),
    columns=pd.MultiIndex.from_product([["EUE", "LOLH"], ["factor", "N_p"]]),
).round(3)

For every method, LOLH gives the wind farm a larger factor than EUE does. EUE responds in every hour in shortfall, while LOLH responds only in hours that the added output can pull out of shortfall, which are the hours with small shortfalls, so the two metrics weigh the hours differently. The four LOLH factors also agree less closely than the four EUE factors, because LOLH changes in jumps. With a counting metric the bisection evaluates both ends of its bracket first, so it takes more than 7 simulations.

Counting metrics are also noisier from one Monte Carlo run to the next. The second table evaluates the ELCC of the wind farm by bisection in five independent runs, with seeds 0 to 4: for EUE and for LOLH at the default 1% step, and for LOLH at 10 MW.

In [ ]:
setups = {
    "EUE, 1% step": MarginalEffectiveLoadCarryingCapability(
        energy_system, simulation, ExpectedUnservedEnergy
    ),
    "LOLH, 1% step": MarginalEffectiveLoadCarryingCapability(
        energy_system, simulation, LossOfLoadHours
    ),
    "LOLH, 10 MW step": MarginalEffectiveLoadCarryingCapability(
        energy_system, simulation, LossOfLoadHours, perturbation_size=10
    ),
}

runs = {}
for name, elcc in setups.items():
    runs[name] = []
    for seed in range(5):
        np.random.seed(seed)
        runs[name].append(elcc.evaluate(wind_farm))

runs = pd.DataFrame(runs, index=[f"run {seed}" for seed in range(5)])
runs.loc["max - min"] = runs.max() - runs.min()
runs.round(3)

The EUE factor barely moves from run to run. The LOLH factor at the 1% step, 1.5 MW for the wind farm, swings widely, and the 10 MW step is much steadier: a small step moves few hours out of shortfall, so the factor rests on few hours. The step also changes the level of the LOLH factor, so state the step along with the factor. In a real study, a counting metric calls for more trials as well.

At the extreme, a step so small that no hour changes status leaves the effect of the perfect resource on LOLH at zero, and the MRI would divide by zero. The classes then raise a `RuntimeWarning` instead of returning a number. The error is also logged, so the message appears twice below.

In [ ]:
too_small = MarginalReliabilityImpact(
    energy_system, simulation, LossOfLoadHours, perturbation_size=0.001
)
np.random.seed(SEED)
try:
    too_small.evaluate(wind_farm)
except RuntimeWarning as warning:
    print("RuntimeWarning:", warning)

## Summary

- `MarginalReliabilityImpact` and `MarginalEffectiveLoadCarryingCapability` accredit units that are already in the system, using their own sampled outages. Both return a dimensionless factor; multiplied by `resources.system_capacity` it gives the accredited capacity in MW.
- Under EUE the two give nearly the same factor for a small step. MRI needs one or two simulations with the resource perturbed (`perturbation_count`), while the marginal ELCC needs a root search, by bisection or secant.
- The default step is 1% of the nameplate of the resources, and `perturbation_size` sets an absolute step instead. Counting metrics such as LOLH need a larger step, and the step belongs with the reported factor.
- Seed NumPy's random number generator before each call for reproducible results, and use more trials than the 200 of this tutorial in a real study.

The Capacity Accreditation section of the reference documentation gives the definitions, the conventions for how resources are perturbed, and the guidance on metrics and step size in full.